In [2]:
import pandas as pd
import numpy as np
import yfinance as yf


In [60]:
tickers = ["^NDX","AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]

prices = yf.download(tickers=tickers ,start="2023-01-01", multi_level_index=False, auto_adjust=True)
prices = prices["Close"]

prices.isnull().sum()
prices.info()


[*********************100%***********************]  8 of 8 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 954 entries, 2023-01-03 to 2026-10-21
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    954 non-null    float64
 1   AMZN    954 non-null    float64
 2   GOOGL   954 non-null    float64
 3   META    954 non-null    float64
 4   MSFT    954 non-null    float64
 5   NVDA    954 non-null    float64
 6   TSLA    954 non-null    float64
 7   ^NDX    954 non-null    float64
dtypes: float64(8)
memory usage: 67.1 KB


In [145]:
#R dziennie stope zwrotu
daily_return = prices.pct_change(1).dropna()
daily_return

#ogolny zwrot
total_return = (1+daily_return).prod() - 1


#Zannualizowana zmiennosc / wahania
annualized_volatility= daily_return.std() * (252**0.5)
annualized_volatility


#zannualizowany zwrot
years = len(daily_return) /252
annualized_return = (1 + total_return) ** (1/years) -1
annualized_return

#Wskaznik sharpa
risk_free_rate = 0
sharpe_ratio = ((annualized_return/years) - risk_free_rate)/annualized_volatility
sharpe_ratio

#najwiekszy dołek
rolling_max = prices.cummax()
drawdown = (prices - rolling_max) / rolling_max
drawdown = drawdown.min()

r2 = daily_return.corr()**2
b , a = np.polyfit(daily_return["^NDX"],daily_return["AAPL"],1)
b

market = daily_return["^NDX"]

def SCL(table):
    company = table
    b , a = np.polyfit(market,company,1)
    tabelka = pd.Series({"alfa": a,"beta" : b})
    return tabelka

scl = daily_return.apply(SCL).T

sharpe_ratio


Ticker
AAPL     0.322733
AMZN     0.279355
GOOGL    0.363768
META     0.394866
MSFT     0.246433
NVDA     0.587602
TSLA     0.185260
^NDX     0.401251
dtype: float64

In [115]:
summary_table = pd.DataFrame({
    "total_return" : total_return,
    "annualized_volatility" : annualized_volatility,
    "sharpe_ratio" : sharpe_ratio,
    "max_drawdown" : drawdown,
    "scl_alfa": scl["alfa"],
    "scl_beta": scl["beta"]
    })

summary_table

,total_return,annualized_volatility,sharpe_ratio,max_drawdown,scl_alfa,scl_beta
Ticker,,,,,,
AAPL,1.802361,0.256635,7.023065,-0.333605,3.752249e-04,0.732916
AMZN,2.056397,0.325337,6.320824,-0.308766,1.432353e-04,1.084577
GOOGL,2.769145,0.305510,9.064013,-0.298059,5.290648e-04,0.918754
META,4.752828,0.393968,12.064009,-0.341508,8.011294e-04,1.170742
MSFT,1.286520,0.262295,4.904853,-0.344984,9.790224e-05,0.793270
NVDA,14.451418,0.478144,30.224010,-0.368810,1.360249e-03,1.719990
TSLA,2.578256,0.572215,4.505747,-0.537657,-2.619197e-05,1.759367
^NDX,1.747022,0.201863,8.654479,-0.229315,8.735704e-19,1.000000


Ticker
AAPL     0.313221
AMZN     0.343702
GOOGL    0.420283
META     0.588305
MSFT     0.244446
NVDA     1.062513
TSLA     0.400897
^NDX     0.306314
dtype: float64